In [5]:
from pathlib import Path
import sys
import importlib
import math

import torch
import pandas as pd
from IPython.display import display
from torch.utils.data import DataLoader

NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

LSTM_ROOT = (NOTEBOOK_DIR / "../..").resolve()
LOADER_DIR = (NOTEBOOK_DIR / "../../../../load/event_log_loader").resolve()
TRAIN_DATA_PATH = (NOTEBOOK_DIR / "../../../../load/encoded_data/helpdesk_all_1_train.pkl").resolve()
TEST_DATA_PATH = (NOTEBOOK_DIR / "../../../../load/encoded_data/helpdesk_all_1_test.pkl").resolve()
TRAIN_EVENT_LOG_PATH = (NOTEBOOK_DIR / "../../../../../transformed_event_logs/Helpdesk_train.csv").resolve()
TEST_EVENT_LOG_PATH = (NOTEBOOK_DIR / "../../../../../transformed_event_logs/Helpdesk_test.csv").resolve()
MODEL_DIR = (NOTEBOOK_DIR / "../training_variational_dropout/Helpdesk").resolve()

for extra_path in (LSTM_ROOT, LOADER_DIR):
    if str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))

import new_event_log_loader
importlib.reload(new_event_log_loader)
from stochasticLSTM.model import StochasticLSTM

for required in (TRAIN_DATA_PATH, TEST_EVENT_LOG_PATH, MODEL_DIR):
    if not required.exists():
        raise FileNotFoundError(required)

DEVICE = torch.device("cpu")

NOTEBOOK_DIR, LSTM_ROOT, LOADER_DIR, DEVICE

(PosixPath('/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_train_LSTM/train_evaluate/LSTM_next_activity_duration/notebooks/evaluation'),
 PosixPath('/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_train_LSTM/train_evaluate/LSTM_next_activity_duration'),
 PosixPath('/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_train_LSTM/load/event_log_loader'),
 device(type='cpu'))

In [37]:
event_log_properties = {
    'case_name' : 'Case ID',
    'concept_name' : 'Activity_start',
    'timestamp_name' : 'Complete Timestamp_start',
    'date_format' : "%Y-%m-%d %H:%M:%S.%f",
    'time_since_case_start_column' : '',
    'time_since_last_event_column' : '',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 1,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.0,
    'window_size' : 'auto',
    'categorical_columns' : ['Activity_start', 'Resource_start'],
    'continuous_columns' : ['seconds_in_day', 'day_in_week', 'duration_seconds'],
    'continuous_positive_columns' : [],
}
event_log_loader = new_event_log_loader.CSV2EventLog(TEST_EVENT_LOG_PATH, **event_log_properties)
test_event_log = event_log_loader.df



selected_cat_attributes = ['Activity_start', 'Resource_start']
selected_num_attributes = ['seconds_in_day', 'day_in_week']
train_event_log_loader = torch.load(TRAIN_DATA_PATH, weights_only=False)
encoder_decoder = train_event_log_loader.encoder_decoder
selected_cat_ids = [i for i, f in enumerate(train_event_log_loader.all_categories[0]) if f[0] in selected_cat_attributes]
selected_num_ids = [i for i, f in enumerate(train_event_log_loader.all_categories[1]) if f[0] in selected_num_attributes]
duration_seconds_id = [i for i, num in enumerate(train_event_log_loader.all_categories[1]) if num[0] == 'duration_seconds'][0]



case_name_col = encoder_decoder.case_name
example_case_id = test_event_log[case_name_col].iloc[100]
example_case_df = test_event_log[test_event_log[case_name_col] == example_case_id].reset_index(drop=True)

encoded_case, _ = encoder_decoder.encode_df(example_case_df)
case_cat_tensors, case_num_tensors, case_ids = encoded_case
sample_idx = len(case_ids) - 1

encoded_case_sample = (
    tuple(t[sample_idx] for t in case_cat_tensors),
    tuple(t[sample_idx] for t in case_num_tensors),
    case_ids[sample_idx],
)

decoded_case_df = encoder_decoder.decode_event(encoded_case_sample)
decoded_case_df = decoded_case_df.replace('nan', pd.NA)

valid_mask = decoded_case_df['Activity_start'].notna() & decoded_case_df['Resource_start'].notna()
decoded_case_df = decoded_case_df[valid_mask].reset_index(drop=True)
display(example_case_df[[case_name_col, 'Activity_start', 'Resource_start', 'duration_seconds']].head())
print(encoded_case)
display(decoded_case_df[[case_name_col, 'Activity_start', 'Resource_start', 'duration_seconds']].tail())

print(f"Example case {example_case_id} produced {len(case_cat_tensors[0])} prefixes; using the final prefix (index {sample_idx}).")

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

Activity_start:   0%|          | 0/1 [00:00<?, ?it/s]

Resource_start:   0%|          | 0/1 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/3 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/1 [00:00<?, ?it/s]

...

duration_seconds:   0%|          | 0/1 [00:00<?, ?it/s]

,Case ID,Activity_start,Resource_start,duration_seconds
0,Case 129,Assign seriousness,Value 1,1553683.0
1,Case 129,Resolve ticket,Value 1,1296022.0


((tensor([[0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 1, 8]]), tensor([[0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 1, 1]])), (tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.5267],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.5267, 1.3824]]), tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.4255],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.4255, -0.6955]]), tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4588],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4588, 0.2528]])), ('Case 129', 'Case 129'))


,Case ID,Activity_start,Resource_start,duration_seconds
0,Case 129,Assign seriousness,Value 1,1553683.0
1,Case 129,Resolve ticket,Value 1,1296022.0


Example case Case 129 produced 2 prefixes; using the final prefix (index 1).


In [25]:
def load_lstm_checkpoint(checkpoint_path: Path) -> StochasticLSTM:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    checkpoint['kwargs'].pop('input_size', None)
    model_instance = StochasticLSTM(**checkpoint['kwargs'])
    model_instance.load_state_dict(checkpoint['model_state_dict'])
    return model_instance.to(DEVICE).eval()

# Select which checkpoint to load
LATEST_MODEL_PATH = max(MODEL_DIR.glob("*.pkl"), key=lambda p: p.stat().st_mtime)

# Load model using the helper defined in a later cell (already available in the notebook)
model = load_lstm_checkpoint(LATEST_MODEL_PATH)

print(f"Loaded model from {LATEST_MODEL_PATH} on device {DEVICE}")

Embeddings:  ModuleList(
  (0): Embedding(13, 16)
  (1): Embedding(23, 16)
)
Total embedding feature size:  32
Input feature size:  34
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1


Loaded model from /home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_train_LSTM/train_evaluate/LSTM_next_activity_duration/notebooks/training_variational_dropout/Helpdesk/model.pkl on device cpu


In [39]:
def prepare_model_payload(encoded_sample):
    cats_full, nums_full, _ = encoded_sample

    # Iterate over all prefix lengths (at least 1 event)
    results = []
    for prefix_len in range(1, cats_full[0].shape[0] + 1):
        # Slice tensors to current prefix length and add batch dimension
        cats_prefix_batched = [tensor[:prefix_len].unsqueeze(0) for tensor in cats_full]
        nums_prefix_batched = [tensor[:prefix_len].unsqueeze(0) for tensor in nums_full]

        selected_cats = [cats_prefix_batched[idx] for idx in selected_cat_ids]
        selected_nums = [nums_prefix_batched[idx] for idx in selected_num_ids]

        model_input = {'cats': selected_cats, 'nums': selected_nums}
        nums_batched = nums_prefix_batched

        with torch.no_grad():
            pred_mean_norm, pred_logvar_norm = model(model_input)

        pred_mean_norm = pred_mean_norm.squeeze().item()
        pred_logvar_norm = pred_logvar_norm.squeeze().item()
        target_norm = nums_batched[duration_seconds_id][0, -1].item()

        duration_scaler = encoder_decoder.continuous_encoders['duration_seconds']
        scale = float(duration_scaler.scale_[0])
        mean_shift = float(duration_scaler.mean_[0])

        pred_mean_seconds = pred_mean_norm * scale + mean_shift
        true_seconds = target_norm * scale + mean_shift
        pred_std_seconds = math.sqrt(math.exp(pred_logvar_norm)) * scale

        results.append(
            {
                'case_id': case_ids[0],  # all from same case
                'prefix_length': prefix_len,
                'predicted_mean_seconds': pred_mean_seconds,
                'predicted_std_seconds': pred_std_seconds,
                'true_duration_seconds': true_seconds,
                'predicted_mean_minutes': pred_mean_seconds / 60.0,
                'true_duration_minutes': true_seconds / 60.0,
            }
        )

    results_df = pd.DataFrame(results)
    #display(results_df.round(2))

    # Keep original return contract for downstream code that expects a single prefix
    # Use the full case as before
    cats_batched_full = [tensor.unsqueeze(0) for tensor in cats_full]
    nums_batched_full = [tensor.unsqueeze(0) for tensor in nums_full]
    selected_cats_full = [cats_batched_full[idx] for idx in selected_cat_ids]
    selected_nums_full = [nums_batched_full[idx] for idx in selected_num_ids]
    return {'cats': selected_cats_full, 'nums': selected_nums_full}, nums_batched_full

model_input, nums_batched = prepare_model_payload(encoded_case_sample)

with torch.no_grad():
    pred_mean_norm, pred_logvar_norm = model(model_input)

pred_mean_norm = pred_mean_norm.squeeze().item()
pred_logvar_norm = pred_logvar_norm.squeeze().item()
target_norm = nums_batched[duration_seconds_id][0, -1].item()

duration_scaler = encoder_decoder.continuous_encoders['duration_seconds']
scale = float(duration_scaler.scale_[0])
mean_shift = float(duration_scaler.mean_[0])

pred_mean_seconds = pred_mean_norm * scale + mean_shift
true_seconds = target_norm * scale + mean_shift
pred_std_seconds = math.sqrt(math.exp(pred_logvar_norm)) * scale

result = pd.DataFrame(
    [
        {
            'case_id': example_case_id,
            'predicted_mean_seconds': pred_mean_seconds,
            'predicted_std_seconds': pred_std_seconds,
            'true_duration_seconds': true_seconds,
        }
    ]
)

result['predicted_mean_minutes'] = result['predicted_mean_seconds'] / 60.0
result['true_duration_minutes'] = result['true_duration_seconds'] / 60.0

display(result.round(2))

,case_id,predicted_mean_seconds,predicted_std_seconds,true_duration_seconds,predicted_mean_minutes,true_duration_minutes
0,Case 129,2603985.8,1428897.79,1296021.98,43399.76,21600.37
